# Putting the Pieces Together: Recreating the PHAT Andomeda Survey
---

## Introduction

In this webinar, you’ll:
- Use astroquery.mast to for search for observations in the MAST Archive
- Learn about MAST’s cloud archive that enables fast data access
- Create an HST mosaic of Andromeda


### Background

The [Panchromatic Hubble Andromeda Treasury (PHAT)](https://archive.stsci.edu/hlsp/phat) was a multi-cycle HST program, designed to efficiently map a third of of the star-forming disk of Andromeda.

![phat brick coverage](phat_coverage.png)

The "efficiency" here comes from using a unique HST observing mode: [parallel observations](https://stsci.recollectcms.com/nodes/view/5271). In short, all HST instruments lie in the same focal plane, so it is possible to use more than one instrument at a time. PHAT takes advantage of this multi-instrument capability to observe in different wavelengths. Even with this clever design, the spatial extent of Andromeda on the sky is quite large. For planning purposes, this program was broken into 23 sections the team named "bricks" (shown above in blue rectangles). Each brick consists of 18 individual HST pointings, taken with the WFC3 and ACS instruments. Today, we'll be creating a mosaic of one of these bricks!

## Imports

We'll need a few packages to accomplish our goal:
* `astropy`, for reading in files, handling astronomy units, and setting a reasonable contrast for our mosaic
* `astroquery` to run our MAST query
* `reproject` to assemble our mosaic

In [ ]:
# run our query
from astroquery.mast import Observations

# read files, use astronomy units,
from astropy.io import fits
from astropy.visualization import ZScaleInterval
import astropy.units as u

from reproject import reproject_interp
from reproject.mosaicking import find_optimal_celestial_wcs, reproject_and_coadd

# display our nice image
import matplotlib.pyplot as plt

import numpy as np

## Running a MAST Query

We want to search MAST.

You can find the searchable parameters on the [CAOM field descriptions](https://mast.stsci.edu/api/v0/_c_a_o_mfields.html) page. If you're in a notebook, you can also run `Observations.get_metadata('observations')`



In [ ]:
Observations.get_metadata('observations')

This is quite useful, if a bit overwhelming. For this notebook, we'll focus on a few key parameters.

here are a few common parameters you may want to use when searching MAST:

| Criterion | Meaning |
|--|--|
|`object_name`| The name of the celestial object: this name will be resolved to coordinates|
|`target_name`| The name of the celestial object **as entered by the proposer**|
|`obs_collection`| Roughly equivalent to mission |

Searching by `object_name` will tend to be a slow filter, since the API must:
1. Query a resolver service (likely Simbad or NED) to transform your input into coordinates
2. Look for observations with a spatial match on your input. This is no small task when the database has over 500 million rows!

TO-DO: talk about setting the resolver manually


Let's run an example query:

In [ ]:
obs = Observations.query_criteria(
    object_name="Trappist-1",
    obs_collection="JWST"
    )

len(obs)

### A Brief Aside: Object vs. Target
There are a fair number of observations. Let's take a look at what the proposers called the targets:

In [ ]:
# get the unique set of target names
set(obs['target_name'])

These results highlight the pitfalls of searching by target name. Sometimes an observer may use a non-standard name, or a the name might not be populated in the database at all!

One useful trick for searching by `target_name` is the use of a wildcard character. In MAST, you can use an asterisk
You can sometimes get around this with wildcard characters:

TO-DO: repeat the above but with `target_name` and a wildcard character. {show an example of a wildcard match before, just in text}

In [ ]:
obs = Observations.query_criteria(
    target_name="Trappist*",
    obs_collection="JWST"
    )

len(obs)

TO-DO: very fast but we're missing 17 observations

## Searching for a Single "Brick"

If we're careful, we can take advantage of the `target_name` trick for very fast searching.

In [ ]:
obs = \
Observations.query_criteria(
    target_name="M31-B01-F*-UVIS",
    wave_region="UV",
    proposal_id=12058,
    project="HST"
                           )

In [ ]:
obs

> "Hey wait a minute. You told me there each brick has 18 observations!"

Indeed. One of our sub-fields is duplicated. Can you figure out which one?

## Cloud Access of MAST Products

The cloud is fast, let's use the cloud.

In [ ]:
prod = Observations.get_product_list(obs)

In [ ]:
prod

SHOW A PREVIEW IMAGE

In [ ]:
filt = Observations.filter_products(prod, mrp_only=True, productSubGroupDescription="DRZ")

In [ ]:
filt

In [ ]:
curis = Observations.get_cloud_uris(filt)

curis

In [ ]:
curis = Observations.get_cloud_uris(filt)

# need "anon":true to anonymously access the cloud files
hdus = [fits.open(f, fsspec_kwargs={"anon": True}) for f in curis]

In [ ]:
# Create a figure on which to plot our data
fig = plt.figure(0, [9, 9])
ax = fig.add_subplot(111)

# Get the primary data from the first fits file
test_data = hdus[0][1].data

# Automatically scale the brightness based on the data
interval = ZScaleInterval(contrast=0.4)
lims = interval.get_limits(test_data)

# Show our data with the scaling from above
ax.imshow(test_data, vmin=lims[0], vmax=lims[1], cmap='grey')

There is a gap in the HST detectors. One way to fill in this gap is to slew the telescope throughout the observations.

## Mosaic Building

Reproject can't read the files from their cloud locations, so we'll need to read them into our memory. This should still be much faster than trying to download the files to your local machine.

Reproject is expecting a particular format of `(data, header)` for each of our observations. We can use Python list comprehension for this:

In [ ]:
hdu_tuples = [(hdu[1].data, hdu[1].header) for hdu in hdus]

Our data preparation is complete!

To combine our images, we need to start by understanding what our output will look like. `find_optimal_celestial_wcs` will select a reasonable "world coordinate system" for image. We need to do this since an plain array of values cannot tell us which way is celestial north. We also need to decide how high-resolution our image should be. Although it is great fun to create 8k images, we'll try to be reasonable in this example. Let's set our resolution to one quarter of an arcsecond.

In [ ]:
wcs, shape = find_optimal_celestial_wcs(hdu_tuples, 
                                        resolution=0.25*u.arcsec
                                       )

We'll use `wcs` later to align our image.

The `shape` tells us the size of our image. Did we choose a reasonable resolution?

In [ ]:
shape

Our image is almost 4k! We're not fully utilizing HST's exquisite sensitivity, but we should still get a nice moasic. 

The `reproject_and_coadd` function will assemble our mosaic for us.
Let's run the code to assemble it:

In [ ]:
array, footprint = reproject_and_coadd(hdu_tuples,
                                       wcs, shape,
                                       reproject_function=reproject_interp,
                                       match_background=True
                                      )

In [ ]:
plt.imshow(footprint)
plt.colorbar()

What we're looking at is, in effect, the number of times HST observed each region.

Our mosaic includes a `0` for locations that have no data from any of our observations. We don't particularly want this, so we can create a mask using our footprint.

In [ ]:
masked_data = np.ma.masked_array(array, footprint==0)

In [ ]:
plt.imsave(fname='test.png', arr=masked_data, vmin=lims[0], vmax=lims[1], origin='lower', cmap='grey')

In [ ]:
# Create the figure for the new map
fig = plt.figure(0, [9, 11])
ax = fig.add_subplot(111, projection=wcs)

# Automatically scale the brightness
interval = ZScaleInterval(contrast=0.4)
lims = interval.get_limits(masked_data)

# Plot the new map
ax.imshow(masked_data, vmin=lims[0], vmax=lims[1], origin='lower', cmap='grey')
ax.set_xlabel('RA')
ax.set_ylabel('Dec')

## About this Notebook

This notebook was written for the 2026 MAST Summer webinar series.

**Author:** Thomas Dutkiewicz<br>
**Keywords:** HST, mosaic, PHAT <br>

***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 